# Figure 7: parent-mode amplitude convergence

The notebook generates the NRSur7dq4 $30+30\,M_\odot$ nonspinning waveform and refits 220--223 from 6--10 $M_f$. Weighted least squares uses the omitted 224 mode and $\epsilon=10^{-22}$. It requires the separately released `tgr` package; no external data or random seed is used.

In [ ]:
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'data').is_dir():
    raise RuntimeError('Run this notebook from the repository root.')

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

# PLOTTING OPTIONS (project-wide convention)
fig_width_pt = 3*246.0
inches_per_pt = 1.0/72.27
golden_mean = (np.sqrt(5)-1.0)/2.0
fig_width = fig_width_pt*inches_per_pt
fig_height = fig_width*golden_mean
fig_size = [fig_width, fig_height]

params = {'axes.labelsize': 24,
          'font.family': 'serif',
          'font.serif': 'Computer Modern Raman',
          'font.size': 24,
          'legend.fontsize': 20,
          'xtick.labelsize': 24,
          'ytick.labelsize': 24,
          'axes.grid': True,
          'text.usetex': True,
          'savefig.dpi': 100,
          'lines.markersize': 14,
          'figure.figsize': fig_size}
mpl.rcParams.update(params)

import lal
import pycbc.waveform
import tgr.nrsurqnm
from tgr.nrsurqnm import (get_qnm_freqtau, least_square_qnmfitting,
                          weighted_least_square_qnmfitting)
print('tgr loaded from:', tgr.nrsurqnm.__file__)

In [ ]:
kwds = {'mass1': 30,
        'mass2': 30,
        'spin1x': 0,
        'spin1y': 0,
        'spin1z': 0,
        'spin2x': 0,
        'spin2y': 0,
        'spin2z': 0,
        'f_lower': 20,
        'f_ref': 20,
        'distance': 400,
        'inclination': 0,
        'delta_t': 1.0 / 4096}

hmodes = pycbc.waveform.get_td_waveform_modes(approximant="NRSur7dq4",
                                              mode_array=['22', '44'], **kwds)
h22 = hmodes[(2, 2)][0] + 1j * hmodes[(2, 2)][1]

In [ ]:
fit_modes = ['220', '221', '222', '223']
omitted_modes = ['224']
EPS_FLOOR = 1e-22   # fiducial; see floor scan below

qnm_par = get_qnm_freqtau(fit_modes + omitted_modes, **kwds)
print('final mass =', qnm_par.final_mass, ' final spin =', qnm_par.final_spin)

Mf_s = qnm_par.final_mass * lal.MTSUN_SI   # remnant mass in seconds
time_m = np.arange(6, 11, 1)               # fit start times in Mf
ref_m = 5                                  # extrapolation reference time in Mf

A_ols, A_wls = {}, {}
for t_m in time_m:
    t0 = t_m * Mf_s
    A_ols[t_m], _, _ = least_square_qnmfitting(fit_modes, qnm_par, t0, h22)
    A_wls[t_m], _, _, info = weighted_least_square_qnmfitting(
        fit_modes, qnm_par, t0, h22, omitted_modes, epsilon_floor=EPS_FLOOR)
    print(f"t0 = {t_m:2d} Mf:  pass-1 |A_224| = {abs(info['A_omitted']['224']):.3e}")

In [ ]:
def extrapolated_amps(A_by_start, mode):
    '''|A| of each fit, extrapolated back to ref_m, as in the original notebook.'''
    out = []
    for t_m in time_m:
        dt = (ref_m - t_m) * Mf_s
        out.append(A_by_start[t_m][mode] * np.exp(-1j * qnm_par.omega(mode) * dt))
    return np.array(out)

def std_over_mean(A_by_start, mode):
    a = np.abs(extrapolated_amps(A_by_start, mode))
    return np.std(a) / np.mean(a)

print(f"{'mode':>6} {'ordinary LS':>14} {'weighted LS':>14} {'improvement':>13}")
conv = {}
for m in fit_modes:
    c_ols = std_over_mean(A_ols, m)
    c_wls = std_over_mean(A_wls, m)
    conv[m] = (c_ols, c_wls)
    print(f"{m:>6} {c_ols:14.5f} {c_wls:14.5f} {c_ols/c_wls:12.1f}x")

In [ ]:
amodes = ['220', '221', '222', '223']
dyfactor = 1e-21
ext_t = np.arange(4, 11, 1)

fig, axes = plt.subplots(4, 2, figsize=(16, 16), sharex=True, sharey='row')

for col, (A_by_start, title) in enumerate([(A_ols, 'ordinary least squares'),
                                           (A_wls, 'weighted least squares')]):
    for row, mode in enumerate(amodes):
        ax = axes[row, col]
        omega = qnm_par.omega(mode)
        for ii, t_m in enumerate(time_m):
            ax.scatter(t_m, np.abs(A_by_start[t_m][mode]) / dyfactor,
                       s=100, c='C' + str(ii), zorder=1)
            dt = (ext_t - t_m) * Mf_s
            A_extrap = A_by_start[t_m][mode] * np.exp(-1j * omega * dt)
            ax.plot(ext_t, np.abs(A_extrap) / dyfactor, c='C' + str(ii), zorder=-1)

        ax.set_yscale('log')
        if col == 0:
            ax.set_ylabel(f'$|A_{{{mode}}}| \\times 10^{{21}}$')
        if row == 0:
            ax.set_title(title, fontsize=22)
            ax.set_ylim(3, 5)
        ax.grid(True, alpha=0.3)

        c = conv[mode][col]
        ax.text(0.03, 0.08, f'std/mean = {c:.4f}',
                transform=ax.transAxes, fontsize=18,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

for ax in axes[-1]:
    ax.set_xlabel('time after merger [$M_\\mathrm{f}$]')
plt.subplots_adjust(hspace=0.1, wspace=0.05)
plt.savefig(ROOT / 'figures/amp_convergence_weighted.pdf', bbox_inches='tight')